# Phase 2.1 — Build the SentiMix master table

One row per tweet, keyed by an immutable `sentence_id`. Produces a proper stratified **train / val / test** split (SentiMix shipped only train+val, but Phase 4 failure analysis needs a labelled test set), reconstructs `text` and per-token `lang_tags` from the CoNLL, and merges in CMI from `sentences.csv`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json

# >>> ADJUST if you put the data elsewhere
SA_DIR   = '/content/drive/MyDrive/codemix_project/data/lince_data/sa_hineng'
OUT_DIR  = '/content/drive/MyDrive/codemix_project/data'

print('SA folder contents:')
for p in sorted(glob.glob(os.path.join(SA_DIR, '*'))):
    print('  ', os.path.basename(p))

## Parse the CoNLL files
SentiMix format: each sentence starts with a `meta<TAB>id<TAB>sentiment` line, then one `token<TAB>langtag` line per token, blank line between sentences. The parser is defensive (handles a missing blank line or a missing tag) and reconstructs `text`, `tokens`, and `lang_tags` in one pass.

In [ ]:
def parse_sentimix_conll(path):
    recs = []
    cur_id = cur_label = None
    toks, tags = [], []
    def flush():
        if cur_id is not None and toks:
            recs.append({'sentence_id': str(cur_id), 'label': cur_label,
                         'tokens': toks[:], 'lang_tags': tags[:]})
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.strip() == '':
                flush(); cur_id = cur_label = None; toks, tags = [], []
                continue
            parts = line.split('\t')
            if parts[0].strip().lower() == 'meta':
                flush(); toks, tags = [], []
                cur_id    = parts[1] if len(parts) > 1 else None
                cur_label = parts[2].strip().lower() if len(parts) > 2 else None
            else:
                toks.append(parts[0])
                tags.append(parts[1] if len(parts) > 1 else 'O')
        flush()
    return recs

# Grab every CoNLL-ish file in the folder (train + val = the full labelled set).
# >>> ADJUST the glob if your files have a different extension/name.
conll_files = [p for p in glob.glob(os.path.join(SA_DIR, '*'))
               if p.lower().endswith(('.txt', '.conll', '.tsv')) and 'sentences' not in os.path.basename(p).lower()]
print('Parsing:', [os.path.basename(p) for p in conll_files])

all_recs = []
for p in conll_files:
    r = parse_sentimix_conll(p)
    print(f'  {os.path.basename(p)}: {len(r)} sentences')
    all_recs.extend(r)

print(f'\nTotal parsed: {len(all_recs)}')
print('\nFirst record (eyeball this):')
ex = all_recs[0]
print('  id   :', ex['sentence_id'])
print('  label:', ex['label'])
print('  text :', ' '.join(ex['tokens'])[:120])
print('  tags :', ex['lang_tags'][:12])

In [ ]:
import pandas as pd

df = pd.DataFrame(all_recs)
df['text'] = df['tokens'].apply(lambda t: ' '.join(t))

# Drop duplicate ids if train/val overlapped for any reason; keep first.
before = len(df)
df = df.drop_duplicates(subset='sentence_id', keep='first').reset_index(drop=True)
print(f'Rows: {before} -> {len(df)} after de-duping on sentence_id')

print('\nLabel distribution:')
print(df['label'].value_counts())

# Stable label<->id mapping. Documented + saved so pred ids map back correctly later.
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
assert set(df['label'].unique()).issubset(LABEL2ID), f"Unexpected labels: {set(df['label'].unique())}"
df['label_id'] = df['label'].map(LABEL2ID)

## Stratified 3-way split (80 / 10 / 10)
Fixed seed so it's reproducible. Stratified on label so all three splits keep the same sentiment balance. `test` is untouched during training/selection — it exists purely for prediction + Phase 4 failure analysis.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
train_df, tmp_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=SEED)
val_df, test_df  = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df['label'], random_state=SEED)

df.loc[train_df.index, 'split'] = 'train'
df.loc[val_df.index,   'split'] = 'val'
df.loc[test_df.index,  'split'] = 'test'

print(df.groupby('split')['label'].value_counts(normalize=True).unstack().round(3))
print('\nSplit sizes:')
print(df['split'].value_counts())

## Merge CMI from `sentences.csv`
Joins the per-tweet Code-Mixing Index you already computed in Phase 0. Column names are detected case-insensitively; adjust if the detection guesses wrong.

In [ ]:
sent_path = os.path.join(SA_DIR, 'sentences.csv')
cmi_df = pd.read_csv(sent_path)
print('sentences.csv columns:', list(cmi_df.columns))

def find_col(cols, *cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c in low: return low[c]
    return None

# >>> ADJUST these two if the auto-detect picks the wrong columns
id_col  = find_col(cmi_df.columns, 'sentence_id', 'id', 'uid', 'tweet_id')
cmi_col = find_col(cmi_df.columns, 'cmi', 'code_mixing_index', 'code-mixing-index')
print(f'Using id column: {id_col!r}, CMI column: {cmi_col!r}')

cmi_small = cmi_df[[id_col, cmi_col]].copy()
cmi_small.columns = ['sentence_id', 'cmi']
cmi_small['sentence_id'] = cmi_small['sentence_id'].astype(str)

n_before = len(df)
df = df.merge(cmi_small, on='sentence_id', how='left')
assert len(df) == n_before, 'Merge changed row count — duplicate ids in sentences.csv?'

missing = df['cmi'].isna().sum()
print(f'\nRows without a CMI match: {missing} / {len(df)}')
if missing:
    print('  (If this is large, the id formats differ between CoNLL and sentences.csv.)')

## Integrity assertions 
If any of these fire, fix the data before training. Silent id drift here corrupts every Phase 4 join downstream.

In [ ]:
# 1. sentence_id is unique
assert df['sentence_id'].is_unique, 'sentence_id not unique!'

# 2. splits are disjoint and cover everything
assert set(df['split']) == {'train', 'val', 'test'}
assert df['split'].notna().all()

# 3. every row has a valid label id
assert df['label_id'].isin([0, 1, 2]).all()

# 4. text is non-empty
assert (df['text'].str.len() > 0).all()

# 5. tokens and lang_tags line up per row
assert (df['tokens'].apply(len) == df['lang_tags'].apply(len)).all()

print('All integrity checks passed.')

In [ ]:
# Serialize list columns to JSON so they survive a CSV round-trip; keep a parquet too.
out = df.copy()
out['tokens']    = out['tokens'].apply(json.dumps)
out['lang_tags'] = out['lang_tags'].apply(json.dumps)

csv_path     = os.path.join(OUT_DIR, 'sa_master.csv')
parquet_path = os.path.join(OUT_DIR, 'sa_master.parquet')
out.to_csv(csv_path, index=False)
df.to_parquet(parquet_path)   # keeps native list columns

with open(os.path.join(OUT_DIR, 'label_mapping.json'), 'w') as f:
    json.dump(LABEL2ID, f, indent=2)

print('Saved:')
print('  ', csv_path)
print('  ', parquet_path)
print('  ', os.path.join(OUT_DIR, 'label_mapping.json'))
print(f'\nMaster table: {len(df)} rows, columns = {list(df.columns)}')